# CNN-LSTM, CNN-BiGRU-Feature-Attention, and CNN-LSTM-Feature-Attention Experiments

This notebook is a runnable copy of the CNN + ResNet + Attention experiment notebook, modified to test three Scientific-Report-style hybrid directions on the IEEE-CIS sequence cache:

1. `cnn_lstm`: CNN local feature extractor for each transaction row, followed by an LSTM temporal encoder.
2. `cnn_bigru_attention`: feature-attention weighting over transaction features, CNN local feature extractor, then BiGRU temporal modelling.
3. `cnn_lstm_attention`: feature-attention weighting over transaction features, CNN local feature extractor, then LSTM temporal modelling.

Important distinction from the previous CNN + ResNet + Attention notebook: here the CNN is **not** used to convolve across timesteps, and attention is **not** used to weight timesteps. The CNN is applied independently to each real transaction row along the feature dimension `F`, and the attention module weights critical input features inside each transaction row. The recurrent layer is responsible for modelling the temporal dimension `T` across the UID history.

Input shape remains `(batch, sequence_length, n_features)`. For the attention cases, a feature gate produces weights with shape `(batch, sequence_length, n_features)` only for real timesteps and reweights each transaction row. The feature CNN then processes only non-padding rows as `(n_real_rows, 1, n_features)`, pools over the feature axis, scatters the embeddings back to `(batch, sequence_length, embedding_dim)`, and passes them to the LSTM/GRU.

Note on the autoencoder idea: the literature sometimes uses an autoencoder as an anomaly/noise filter before the downstream predictor. In fraud detection, filtering high-reconstruction-error rows can accidentally remove true fraud cases, so the autoencoder filter is implemented as an optional ablation and disabled by default (`AE_FILTER_ENABLE = False`).


## 1. Imports and Configuration

In [1]:
import os, gc, math, time, warnings, copy
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

try:
    from tqdm.auto import tqdm
except Exception:
    class _NoTqdm:
        def __init__(self, iterable=None, **kwargs):
            self.iterable = iterable
        def __iter__(self):
            return iter(self.iterable)
        def set_postfix(self, *args, **kwargs):
            pass
    def tqdm(iterable=None, **kwargs):
        return _NoTqdm(iterable)

# ----- General experiment config -----
WINDOW              = 20
MIN_TRAIN_MONTHS    = 3
BATCH               = 512
EPOCHS              = 30
LR                  = 1e-3
WEIGHT_DECAY        = 2e-4
GRAD_CLIP           = 1.0
EARLY_STOP_PATIENCE = 6
SEED                = 42
N_SEEDS             = 3       # keep 3 seeds for comparability with previous CNN runs
USE_POS_WEIGHT      = False
PREDICT_TEST        = False   # tuning mode: save OOF only, skip expensive test inference
PROGRESS_BAR        = True    # show batch-level train/validation progress
PROGRESS_UPDATE_EVERY = 25    # reduce tqdm overhead by updating postfix every N batches

# Set to None for auto, or one of: "cpu", "cuda", "mps".
# Useful on Kaggle when GPU quota is exhausted.
DEVICE_OVERRIDE     = None
CPU_NUM_THREADS     = min(8, os.cpu_count() or 1)

# ----- Model width/depth config -----
CNN_FEATURE_CHANNELS= 32      # channels while convolving over feature axis
CNN_EMBED_DIM       = 128     # per-transaction embedding passed to LSTM/GRU
N_RESBLOCKS         = 2       # keep >=2 for comparability; set 1 for a faster ablation
RNN_HIDDEN          = 128
RNN_LAYERS          = 2
RNN_DROPOUT         = 0.3
RNN_BIDIRECTIONAL   = True    # matches Time_Series_LSTM_left_pad_bidirectional_v3 style
USE_PACKED_RNN      = True    # avoids recurrent compute over padded timesteps
FEATURE_ATTN_HIDDEN = 128
ATTN_DROPOUT        = 0.2
STATIC_HIDDEN       = 64
USE_STATIC_TOWER    = True
USE_MEAN_MAX_POOL   = True

# ----- Optional autoencoder pre-filter for CNN-BiGRU-attention ablation -----
# Disabled by default because filtering high reconstruction error can remove true fraud examples.
AE_FILTER_ENABLE    = False
AE_FILTER_QUANTILE  = 0.995
AE_EPOCHS           = 2
AE_BATCH            = 2048
AE_MAX_FIT_ROWS     = 100_000

MODEL_CASES = [
    {"name": "cnn_lstm", "kind": "cnn_lstm", "use_feature_attention": False, "rnn_type": "lstm", "use_ae_filter": False},
    {"name": "cnn_bigru_attention", "kind": "cnn_bigru_attention", "use_feature_attention": True, "rnn_type": "gru", "use_ae_filter": True},
    {"name": "cnn_lstm_attention", "kind": "cnn_lstm_attention", "use_feature_attention": True, "rnn_type": "lstm", "use_ae_filter": False},
]

RUN_CASES = [case["name"] for case in MODEL_CASES]
OUTPUT_DIR = Path("cnn_hybrid_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----- Reproducibility -----
torch.manual_seed(SEED)
np.random.seed(SEED)

# ----- Device selection -----
def resolve_device(override=None):
    if override is not None:
        requested = str(override).lower().strip()
        if requested == 'cuda' and torch.cuda.is_available():
            return torch.device('cuda')
        if requested == 'mps' and torch.backends.mps.is_available():
            return torch.device('mps')
        if requested == 'cpu':
            return torch.device('cpu')
        print(f"Requested DEVICE_OVERRIDE={override!r} is unavailable. Falling back to auto device selection.")

    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

device = resolve_device(DEVICE_OVERRIDE)
if device.type == 'cpu':
    torch.set_num_threads(CPU_NUM_THREADS)

print(f'PyTorch {torch.__version__}  device={device}')
if device.type == 'cpu':
    print(f'CPU threads: {torch.get_num_threads()}')
print('Cases:', RUN_CASES)
print('PREDICT_TEST:', PREDICT_TEST)


PyTorch 2.10.0+cu128  device=cuda
Cases: ['cnn_lstm', 'cnn_bigru_attention', 'cnn_lstm_attention']
PREDICT_TEST: False


## 2. Load Sequence Cache

The notebook first tries the Kaggle `.npy` cache path. If that is not available, it falls back to the local project cache. If neither `.npy` cache exists, it falls back to `split_data.npz`.

In [2]:
N_NPY_REQUIRED = [
    "X_train_seq", "X_test_seq", "L_train", "L_test", "train_order", "test_order",
    "train_pos", "test_pos", "y_aligned", "dt_m_aligned",
]

CANDIDATE_NPY_DIRS = [
    Path("/kaggle/input/datasets/bachhoviet/split-data-npy"),
    Path("/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/split_data_npy"),
]

CANDIDATE_NPZ_FILES = [
    Path("/kaggle/input/datasets/bachhoviet/split_data.npz"),
    Path("/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/split_data.npz"),
]

def _has_npy_cache(d: Path) -> bool:
    return d.exists() and all((d / f"{name}.npy").exists() for name in N_NPY_REQUIRED)

def load_from_npy_dir(d: Path):
    def load(name, mmap=True):
        return np.load(d / f"{name}.npy", mmap_mode='r' if mmap else None)
    return {
        "X_train_seq":  load("X_train_seq", mmap=True),
        "X_test_seq":   load("X_test_seq", mmap=True),
        "L_train":      load("L_train", mmap=False),
        "L_test":       load("L_test", mmap=False),
        "train_order":  load("train_order", mmap=False),
        "test_order":   load("test_order", mmap=False),
        "train_pos":    load("train_pos", mmap=False),
        "test_pos":     load("test_pos", mmap=False),
        "y_aligned":    load("y_aligned", mmap=False),
        "dt_m_aligned": load("dt_m_aligned", mmap=False),
    }

def load_sequence_cache():
    for d in CANDIDATE_NPY_DIRS:
        if _has_npy_cache(d):
            print(f"Loading .npy sequence cache from: {d}")
            return load_from_npy_dir(d)
    for p in CANDIDATE_NPZ_FILES:
        if p.exists():
            print(f"Loading .npz sequence cache from: {p}")
            data = np.load(p)
            return {name: data[name] for name in N_NPY_REQUIRED}
    raise FileNotFoundError("No split-data sequence cache found. Update CANDIDATE_NPY_DIRS or CANDIDATE_NPZ_FILES.")

cache = load_sequence_cache()
X_train_seq  = cache["X_train_seq"]
X_test_seq   = cache["X_test_seq"]
L_train      = cache["L_train"].astype("int64")
L_test       = cache["L_test"].astype("int64")
train_order  = cache["train_order"]
test_order   = cache["test_order"]
train_pos    = cache["train_pos"]
test_pos     = cache["test_pos"]
y_aligned    = cache["y_aligned"].astype("float32")
dt_m_aligned = cache["dt_m_aligned"]

N_FEATURES = X_train_seq.shape[2]
print(f"X_train_seq: {X_train_seq.shape} {X_train_seq.dtype}")
print(f"X_test_seq : {X_test_seq.shape} {X_test_seq.dtype}")
print(f"L_train    : {L_train.shape} {L_train.dtype}")
print(f"y_aligned  : {y_aligned.shape} {y_aligned.dtype}, positive rate={y_aligned.mean():.4f}")


Loading .npy sequence cache from: /kaggle/input/datasets/bachhoviet/split-data-npy
X_train_seq: (590540, 20, 234) float16
X_test_seq : (506691, 20, 234) float16
L_train    : (590540,) int64
y_aligned  : (590540,) float32, positive rate=0.0350


## 3. Dataset and DataLoader

In [3]:
class WindowDataset(Dataset):
    def __init__(self, X, lengths, y=None):
        self.X = torch.from_numpy(X)
        self.lengths = torch.from_numpy(lengths.astype('int64'))
        self.y = None if y is None else torch.from_numpy(y.astype('float32'))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        if self.y is None:
            return self.X[i], self.lengths[i]
        return self.X[i], self.lengths[i], self.y[i]


def make_loader(X, lengths, y, batch_size, shuffle):
    return DataLoader(
        WindowDataset(X, lengths, y),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )


## 4. Hybrid CNN/RNN/Attention Models

All models use left-padded sequence windows. Real timesteps are at the right side of the sequence, so the mask is computed from the end of the window. The most recent transaction is always `x[:, -1, :]`.

In [4]:
def left_padded_real_mask(lengths: torch.Tensor, T: int, device=None) -> torch.Tensor:
    """Return True for real timesteps in a left-padded sequence."""
    if device is None:
        device = lengths.device
    lengths = lengths.to(device=device, dtype=torch.long).clamp(min=1, max=T)
    idx = torch.arange(T, device=device).unsqueeze(0)
    pos_from_end = T - 1 - idx
    return pos_from_end < lengths.unsqueeze(1)


def right_padded_real_mask(lengths: torch.Tensor, T: int, device=None) -> torch.Tensor:
    """Return True for real timesteps in a right-padded sequence."""
    if device is None:
        device = lengths.device
    lengths = lengths.to(device=device, dtype=torch.long).clamp(min=1, max=T)
    idx = torch.arange(T, device=device).unsqueeze(0)
    return idx < lengths.unsqueeze(1)


def left_to_right_padded(h: torch.Tensor, lengths: torch.Tensor):
    """
    Convert left-padded sequence embeddings to right-padded embeddings.

    pack_padded_sequence expects real timesteps to start at index 0. The sequence cache is
    left-padded, so real timesteps are at the end of the window and must be shifted left
    before packed LSTM/GRU execution.
    """
    B, T, E = h.shape
    device = h.device
    lengths = lengths.to(device=device, dtype=torch.long).clamp(min=1, max=T)
    dst_pos = torch.arange(T, device=device).unsqueeze(0).expand(B, T)
    src_pos = (T - lengths.unsqueeze(1) + dst_pos).clamp(min=0, max=T - 1)
    out = h.gather(1, src_pos.unsqueeze(-1).expand(B, T, E))
    real = dst_pos < lengths.unsqueeze(1)
    out = out.masked_fill(~real.unsqueeze(-1), 0.0)
    return out, real


def masked_mean_max(h: torch.Tensor, real: torch.Tensor):
    """Masked mean and max over temporal dimension."""
    mask_f = real.unsqueeze(-1).float()
    mean = (h * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
    neg_inf = torch.finfo(h.dtype).min
    maxv = h.masked_fill(~real.unsqueeze(-1), neg_inf).max(dim=1).values
    return mean, maxv


class FeatureAttentionGate(nn.Module):
    """
    Feature-wise attention gate.

    Input : x with shape (B, T, F)
    Output: reweighted x and feature weights with shape (B, T, F)

    This attention operates over input features inside each transaction row, not over timesteps.
    Padding timesteps are skipped instead of being processed by the MLP.
    """
    def __init__(self, n_features, hidden=FEATURE_ATTN_HIDDEN, drop=ATTN_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(hidden, n_features),
            nn.Sigmoid(),
        )

    def forward(self, x, real=None):
        B, T, F = x.shape
        flat = x.reshape(B * T, F)

        if real is None:
            weights = self.net(flat).view(B, T, F)
            return x * weights, weights

        real_idx = real.reshape(-1).nonzero(as_tuple=False).squeeze(1)
        weights_flat = flat.new_zeros(B * T, F)
        if real_idx.numel() > 0:
            weights_real = self.net(flat.index_select(0, real_idx))
            weights_flat = weights_flat.index_copy(0, real_idx, weights_real)
        weights = weights_flat.view(B, T, F)
        return x * weights, weights


class FeatureResBlock1D(nn.Module):
    """Residual Conv1D block over the feature axis of a single transaction row."""
    def __init__(self, c: int, k: int = 3, drop: float = 0.1):
        super().__init__()
        p = k // 2
        self.conv1 = nn.Conv1d(c, c, kernel_size=k, padding=p)
        self.bn1   = nn.BatchNorm1d(c)
        self.conv2 = nn.Conv1d(c, c, kernel_size=k, padding=p)
        self.bn2   = nn.BatchNorm1d(c)
        self.drop  = nn.Dropout(drop)
        self.act   = nn.ReLU()

    def forward(self, x):
        identity = x
        out = self.act(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        return self.act(out + identity)


class CNNFeatureExtractor(nn.Module):
    """
    CNN local feature extractor.

    Input : (B, T, F)
    Step  : select only real timesteps, reshape to (N_real, 1, F),
            so Conv1D slides along feature dimension F.
    Output: (B, T, embed_dim), one feature-interaction embedding per real transaction.
    """
    def __init__(self, n_features, feature_channels=CNN_FEATURE_CHANNELS,
                 embed_dim=CNN_EMBED_DIM, n_resblocks=N_RESBLOCKS, drop=ATTN_DROPOUT):
        super().__init__()
        self.n_features = n_features
        self.embed_dim = embed_dim
        self.stem = nn.Sequential(
            nn.Conv1d(1, feature_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(feature_channels),
            nn.ReLU(),
        )
        self.resblocks = nn.Sequential(*[
            FeatureResBlock1D(feature_channels, k=3, drop=drop)
            for _ in range(n_resblocks)
        ])
        self.proj = nn.Sequential(
            nn.Linear(feature_channels * 2, embed_dim),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, x, real=None):
        B, T, F = x.shape
        flat = x.reshape(B * T, F)

        if real is None:
            row_x = flat.unsqueeze(1).contiguous()       # (B*T, 1, F)
            h = self.stem(row_x)
            h = self.resblocks(h)
            mean_pool = h.mean(dim=2)
            max_pool = h.max(dim=2).values
            return self.proj(torch.cat([mean_pool, max_pool], dim=1)).view(B, T, self.embed_dim)

        real_idx = real.reshape(-1).nonzero(as_tuple=False).squeeze(1)
        row_emb_flat = flat.new_zeros(B * T, self.embed_dim)
        if real_idx.numel() == 0:
            return row_emb_flat.view(B, T, self.embed_dim)

        row_x = flat.index_select(0, real_idx).unsqueeze(1).contiguous()  # (N_real, 1, F)
        h = self.stem(row_x)                                             # (N_real, C, F)
        h = self.resblocks(h)                                            # (N_real, C, F)
        mean_pool = h.mean(dim=2)                                        # (N_real, C)
        max_pool = h.max(dim=2).values                                   # (N_real, C)
        row_emb = self.proj(torch.cat([mean_pool, max_pool], dim=1))      # (N_real, E)
        row_emb_flat = row_emb_flat.index_copy(0, real_idx, row_emb)
        return row_emb_flat.view(B, T, self.embed_dim)


class StaticTower(nn.Module):
    def __init__(self, n_features, static_hidden=STATIC_HIDDEN, drop=RNN_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(128, static_hidden), nn.ReLU(),
        )

    def forward(self, last_row):
        return self.net(last_row)


class CNNRNNHybridClassifier(nn.Module):
    """
    Hybrid model where feature attention weights critical input features,
    CNN extracts local feature interactions per transaction row,
    then LSTM/GRU models the temporal sequence across rows.
    """
    def __init__(self, n_features, rnn_type='lstm', use_feature_attention=False,
                 cnn_embed_dim=CNN_EMBED_DIM, rnn_hidden=RNN_HIDDEN, rnn_layers=RNN_LAYERS,
                 bidirectional=RNN_BIDIRECTIONAL, drop=RNN_DROPOUT,
                 use_static_tower=USE_STATIC_TOWER, use_mean_max_pool=USE_MEAN_MAX_POOL,
                 use_packed_rnn=USE_PACKED_RNN):
        super().__init__()
        self.rnn_type = rnn_type.lower()
        self.use_feature_attention = use_feature_attention
        self.use_static_tower = use_static_tower
        self.use_mean_max_pool = use_mean_max_pool
        self.use_packed_rnn = use_packed_rnn

        if use_feature_attention:
            self.feature_attention = FeatureAttentionGate(n_features)

        self.feature_cnn = CNNFeatureExtractor(n_features, embed_dim=cnn_embed_dim)

        rnn_cls = nn.GRU if self.rnn_type == 'gru' else nn.LSTM
        self.rnn = rnn_cls(
            input_size=cnn_embed_dim,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            dropout=drop if rnn_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        rnn_out_dim = rnn_hidden * (2 if bidirectional else 1)

        pool_dim = rnn_out_dim
        if use_mean_max_pool:
            pool_dim += 2 * rnn_out_dim

        if use_static_tower:
            self.static_mlp = StaticTower(n_features, static_hidden=STATIC_HIDDEN, drop=drop)
            pool_dim += STATIC_HIDDEN

        self.head = nn.Sequential(
            nn.Linear(pool_dim, 64), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(64, 1),
        )

    def forward(self, x, lengths):
        B, T, _ = x.shape
        lengths = lengths.to(device=x.device, dtype=torch.long).clamp(min=1, max=T)
        left_real = left_padded_real_mask(lengths, T, device=x.device)

        if self.use_feature_attention:
            x_used, feature_weights = self.feature_attention(x, real=left_real)
        else:
            x_used = x

        # For left-padded windows, the current transaction is always at index T-1.
        # For attention cases, the static tower sees the feature-weighted current row.
        last_row = x_used[:, -1, :]

        # CNN extracts local feature interactions only for real transaction rows.
        h_left = self.feature_cnn(x_used, real=left_real)  # (B, T, CNN_EMBED_DIM), left-padded

        if self.use_packed_rnn:
            h, rnn_real = left_to_right_padded(h_left, lengths)
            packed = pack_padded_sequence(
                h,
                lengths.detach().cpu(),
                batch_first=True,
                enforce_sorted=False,
            )
            packed_out, _ = self.rnn(packed)
            rnn_out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=T)
            rnn_out = rnn_out.masked_fill(~rnn_real.unsqueeze(-1), 0.0)
            last_idx = (lengths - 1).clamp(min=0)
            last_hidden = rnn_out[torch.arange(B, device=x.device), last_idx, :]
        else:
            rnn_out, _ = self.rnn(h_left)
            rnn_real = left_real
            rnn_out = rnn_out.masked_fill(~rnn_real.unsqueeze(-1), 0.0)
            last_hidden = rnn_out[:, -1, :]

        parts = [last_hidden]

        if self.use_mean_max_pool:
            mean, maxv = masked_mean_max(rnn_out, rnn_real)
            parts.extend([mean, maxv])

        if self.use_static_tower:
            parts.append(self.static_mlp(last_row))

        feat = torch.cat(parts, dim=1)
        return self.head(feat).squeeze(-1)


def make_model(case, n_features):
    kind = case['kind']
    if kind == 'cnn_lstm':
        return CNNRNNHybridClassifier(n_features, rnn_type='lstm', use_feature_attention=False)
    if kind == 'cnn_bigru_attention':
        return CNNRNNHybridClassifier(n_features, rnn_type='gru', use_feature_attention=True, bidirectional=True)
    if kind == 'cnn_lstm_attention':
        return CNNRNNHybridClassifier(n_features, rnn_type='lstm', use_feature_attention=True)
    raise ValueError(f"Unknown model kind: {kind}")


## 5. Optional Autoencoder Filter

This is available for the CNN-BiGRU-attention ablation. It is disabled by default because fraud examples are often exactly the unusual examples we do not want to remove. If enabled, it trains a small reconstruction model on the fold training set and keeps rows below a reconstruction-error quantile.

In [5]:
class RowFeatureAutoencoder(nn.Module):
    """Dense autoencoder over individual real transaction rows."""
    def __init__(self, n_features, hidden=256, latent=64, drop=0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, latent), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, hidden), nn.ReLU(),
            nn.Linear(hidden, n_features),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def _real_rows_numpy(X, L, max_rows=None, seed=SEED):
    rows = []
    total = len(X)
    indices = np.arange(total)
    if max_rows is not None and total > max_rows:
        rng = np.random.default_rng(seed)
        indices = rng.choice(total, size=max_rows, replace=False)
    for i in indices:
        length = int(L[i])
        if length > 0:
            rows.append(X[i, -length:, :])  # left-padded, real rows are at the right end
    if not rows:
        return np.empty((0, X.shape[2]), dtype=np.float32)
    return np.concatenate(rows, axis=0).astype('float32')


def fit_autoencoder_filter(X, L, n_features, device, quantile=AE_FILTER_QUANTILE):
    """
    Optional row-level anomaly filter.
    Trains on real transaction rows, then scores each sequence by its max row reconstruction error.
    """
    n = len(X)
    if n == 0:
        return np.ones(0, dtype=bool)

    train_rows = _real_rows_numpy(X, L, max_rows=AE_MAX_FIT_ROWS)
    if len(train_rows) == 0:
        return np.ones(n, dtype=bool)

    ae = RowFeatureAutoencoder(n_features).to(device)
    opt = torch.optim.AdamW(ae.parameters(), lr=1e-3, weight_decay=1e-4)
    row_tensor = torch.from_numpy(train_rows)
    row_loader = DataLoader(row_tensor, batch_size=AE_BATCH, shuffle=True, num_workers=0)

    for epoch in range(1, AE_EPOCHS + 1):
        ae.train(); running, seen = 0.0, 0
        for xb in row_loader:
            xb = xb.to(device=device, dtype=torch.float32)
            opt.zero_grad(set_to_none=True)
            rec = ae(xb)
            loss = (rec - xb).pow(2).mean()
            loss.backward()
            opt.step()
            running += loss.item() * xb.size(0); seen += xb.size(0)
        print(f"      AE epoch {epoch}/{AE_EPOCHS} row_loss={running/max(seen,1):.5f}")

    seq_errors = np.zeros(n, dtype=np.float32)
    ae.eval()
    with torch.no_grad():
        for start in range(0, n, 1024):
            end = min(start + 1024, n)
            batch_errs = []
            for i in range(start, end):
                length = int(L[i])
                if length <= 0:
                    batch_errs.append(0.0)
                    continue
                rows = torch.from_numpy(np.asarray(X[i, -length:, :], dtype=np.float32)).to(device)
                rec = ae(rows)
                row_err = (rec - rows).pow(2).mean(dim=1)
                batch_errs.append(float(row_err.max().cpu()))
            seq_errors[start:end] = np.array(batch_errs, dtype=np.float32)

    cutoff = np.quantile(seq_errors, quantile)
    keep = seq_errors <= cutoff
    print(f"      AE row-filter cutoff={cutoff:.6f}, kept={keep.mean():.2%} ({keep.sum():,}/{len(keep):,})")
    del ae
    if device.type == 'mps': torch.mps.empty_cache()
    gc.collect()
    return keep


## 6. Expanding-Window CV and Training Loop

In [6]:
def expanding_month_folds(months_array, min_train_months=MIN_TRAIN_MONTHS):
    months = sorted(np.unique(months_array).tolist())
    for vm in months[min_train_months:]:
        tm = [m for m in months if m < vm]
        ti = np.flatnonzero(np.isin(months_array, tm))
        vi = np.flatnonzero(months_array == vm)
        yield (vm, tm, ti, vi)


def train_one_fold(case, X_tr, L_tr, y_tr, X_va, L_va, y_va, n_features,
                   epochs, batch, lr, weight_decay, device,
                   early_stop_patience, grad_clip, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    if case.get('use_ae_filter', False) and AE_FILTER_ENABLE:
        print('      fitting autoencoder filter...', flush=True)
        keep = fit_autoencoder_filter(X_tr, L_tr, n_features, device)
        X_tr, L_tr, y_tr = X_tr[keep], L_tr[keep], y_tr[keep]
        print(f"      after AE filter: train rows={len(X_tr):,}, positive rate={y_tr.mean():.4f}", flush=True)

    model = make_model(case, n_features).to(device)

    if USE_POS_WEIGHT:
        pos = float((y_tr == 1).sum())
        neg = float(len(y_tr) - pos)
        pw = torch.tensor([np.sqrt(neg / max(pos, 1.0))], device=device, dtype=torch.float32)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        loss_fn = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_loader = make_loader(X_tr, L_tr, y_tr, batch_size=batch, shuffle=True)
    val_loader   = make_loader(X_va, L_va, y_va, batch_size=batch, shuffle=False)

    steps = max(1, math.ceil(len(X_tr) / batch))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=steps,
        pct_start=0.1, anneal_strategy='cos',
    )

    best_auc, best_state, best_val_preds, bad = -1.0, None, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time(); running, n_seen = 0.0, 0
        print(
            f"   ep {epoch:>2}/{epochs} start "
            f"(train_batches={len(train_loader):,}, valid_batches={len(val_loader):,})",
            flush=True,
        )

        train_iter = tqdm(
            train_loader,
            total=len(train_loader),
            desc=f"{case['name']} seed{seed} ep{epoch}/{epochs} train",
            leave=False,
            dynamic_ncols=True,
            disable=not PROGRESS_BAR,
        )
        for step_idx, (xb, lb, yb) in enumerate(train_iter, start=1):
            xb = xb.to(device=device, dtype=torch.float32)
            lb = lb.to(device=device, dtype=torch.long)
            yb = yb.to(device=device, dtype=torch.float32)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb, lb).squeeze(-1)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step(); scheduler.step()

            running += loss.item() * xb.size(0)
            n_seen += xb.size(0)
            if step_idx == 1 or step_idx % PROGRESS_UPDATE_EVERY == 0 or step_idx == len(train_loader):
                current_lr = scheduler.get_last_lr()[0]
                train_iter.set_postfix(loss=f"{running / max(n_seen, 1):.4f}", lr=f"{current_lr:.2e}")
        train_loss = running / max(n_seen, 1)

        model.eval(); preds = []
        val_iter = tqdm(
            val_loader,
            total=len(val_loader),
            desc=f"{case['name']} seed{seed} ep{epoch}/{epochs} valid",
            leave=False,
            dynamic_ncols=True,
            disable=not PROGRESS_BAR,
        )
        with torch.no_grad():
            for xb, lb, _ in val_iter:
                xb = xb.to(device=device, dtype=torch.float32)
                lb = lb.to(device=device, dtype=torch.long)
                logits = model(xb, lb).squeeze(-1)
                preds.append(torch.sigmoid(logits).cpu().numpy())
        val_preds = np.concatenate(preds)
        val_auc = roc_auc_score(y_va, val_preds)

        print(f"   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)", flush=True)
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())
            best_val_preds = val_preds
            bad = 0
        else:
            bad += 1
            if bad >= early_stop_patience:
                print(f"   early stop at epoch {epoch}", flush=True)
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val_preds, best_auc, model


## 7. Smoke Test All Three Models

In [7]:
xb = torch.randn(8, WINDOW, N_FEATURES, device=device)
lb = torch.randint(1, WINDOW + 1, (8,), device=device)
for case in MODEL_CASES:
    m = make_model(case, N_FEATURES).to(device)
    out = m(xb, lb)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{case['name']:<24} output={tuple(out.shape)} params={n_params:,}")
    del m
if device.type == 'mps': torch.mps.empty_cache()
del xb, lb
gc.collect()


cnn_lstm                 output=(8,) params=835,649
cnn_bigru_attention      output=(8,) params=731,051
cnn_lstm_attention       output=(8,) params=895,915


37

## 8. Run Cases Separately Without Test Prediction

The helper below keeps the same 3-fold, 3-seed protocol, but each model case is launched from its own cell. By default `PREDICT_TEST = False`, so the run saves OOF validation predictions and skips expensive repeated test-set inference during tuning.

If final submission predictions are needed later, set `PREDICT_TEST = True` before running a selected case.


In [8]:
fold_specs = list(expanding_month_folds(dt_m_aligned, MIN_TRAIN_MONTHS))
print(f"{len(fold_specs)} folds; {N_SEEDS} seeds per fold", flush=True)
print(f"Test prediction enabled: {PREDICT_TEST}", flush=True)

case_lookup = {case['name']: case for case in MODEL_CASES}
experiment_summaries = []


def clear_device_cache():
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    elif device.type == 'mps':
        torch.mps.empty_cache()
    gc.collect()


def predict_test_for_model(model):
    test_loader = make_loader(X_test_seq, L_test, y=None, batch_size=BATCH, shuffle=False)
    tps = []
    model.eval()
    test_iter = tqdm(
        test_loader,
        total=len(test_loader),
        desc="test inference",
        leave=False,
        dynamic_ncols=True,
        disable=not PROGRESS_BAR,
    )
    with torch.no_grad():
        for xb, lb in test_iter:
            xb = xb.to(device=device, dtype=torch.float32)
            lb = lb.to(device=device, dtype=torch.long)
            logits = model(xb, lb).squeeze(-1)
            tps.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(tps)


def run_experiment_case(case_name, predict_test=PREDICT_TEST):
    if case_name not in case_lookup:
        raise ValueError(f"Unknown case_name={case_name!r}. Available: {list(case_lookup)}")

    case = case_lookup[case_name]
    print("\n" + "=" * 90, flush=True)
    print(f"RUNNING CASE: {case_name}", flush=True)
    print(f"predict_test={predict_test}", flush=True)
    print("=" * 90, flush=True)

    oof = np.full(len(X_train_seq), np.nan, dtype=np.float32)
    test_preds = np.zeros(len(X_test_seq), dtype=np.float32) if predict_test else None
    fold_aucs = []

    for fold, (vm, tm, idxT, idxV) in enumerate(fold_specs):
        print(f"\n=== {case_name} | Fold {fold}: train {tm} -> validate {vm} "
              f"(train={len(idxT):,}, valid={len(idxV):,}) ===")

        seed_val_preds = []
        seed_test_preds = [] if predict_test else None

        for s in range(N_SEEDS):
            seed = SEED + s
            print(f"-- seed {seed} --", flush=True)
            vp, va, model = train_one_fold(
                case,
                X_train_seq[idxT], L_train[idxT], y_aligned[idxT],
                X_train_seq[idxV], L_train[idxV], y_aligned[idxV],
                n_features=N_FEATURES, epochs=EPOCHS, batch=BATCH,
                lr=LR, weight_decay=WEIGHT_DECAY, device=device,
                early_stop_patience=EARLY_STOP_PATIENCE,
                grad_clip=GRAD_CLIP, seed=seed,
            )
            seed_val_preds.append(vp)

            if predict_test:
                seed_test_preds.append(predict_test_for_model(model))

            del model
            clear_device_cache()

        fold_val = np.mean(seed_val_preds, axis=0)
        fold_auc = roc_auc_score(y_aligned[idxV], fold_val)
        print(f"   fold AUC (seed-avg) = {fold_auc:.4f}")

        fold_aucs.append((int(vm), float(fold_auc)))
        oof[idxV] = fold_val

        if predict_test:
            test_preds += np.mean(seed_test_preds, axis=0)

    if predict_test and len(fold_specs):
        test_preds /= len(fold_specs)

    validated = ~np.isnan(oof)
    overall_auc = roc_auc_score(y_aligned[validated], oof[validated])
    print(f"\n=== {case_name} OOF AUC (validated months only) = {overall_auc:.4f} ===")
    print(f"   per-fold: {fold_aucs}")

    oof_df = pd.DataFrame({
        'TransactionID': train_order,
        f'oof_{case_name}': oof,
        'isFraud': y_aligned,
    })

    oof_path = OUTPUT_DIR / f"oof_{case_name}.csv"
    oof_df.to_csv(oof_path, index=False)
    print(f"Saved {oof_path}")

    test_path = None
    if predict_test:
        test_df = pd.DataFrame({
            'TransactionID': test_order,
            f'pred_{case_name}': test_preds,
        })
        test_path = OUTPUT_DIR / f"test_pred_{case_name}.csv"
        test_df.to_csv(test_path, index=False)
        print(f"Saved {test_path}")
    else:
        print("Skipped test prediction. Set PREDICT_TEST=True for final submission inference.")

    summary_row = {
        'case': case_name,
        'overall_oof_auc': float(overall_auc),
        'fold_aucs': repr(fold_aucs),
        'oof_path': str(oof_path),
        'test_path': '' if test_path is None else str(test_path),
        'predict_test': bool(predict_test),
        'n_seeds': int(N_SEEDS),
        'n_folds': int(len(fold_specs)),
        'n_resblocks': int(N_RESBLOCKS),
        'skip_padded_cnn_rows': True,
        'use_packed_rnn': bool(USE_PACKED_RNN),
        'ae_filter_enabled': bool(AE_FILTER_ENABLE and case.get('use_ae_filter', False)),
    }

    summary_path = OUTPUT_DIR / f"summary_{case_name}.csv"
    pd.DataFrame([summary_row]).to_csv(summary_path, index=False)
    print(f"Saved {summary_path}")

    experiment_summaries.append(summary_row)
    return summary_row


3 folds; 3 seeds per fold
Test prediction enabled: False


### Case 1: CNN-LSTM

CNN extracts local feature interactions for each real transaction row. A bidirectional LSTM models the temporal sequence. Test prediction is skipped in this tuning run.


In [9]:
cnn_lstm_summary = run_experiment_case("cnn_lstm", predict_test=False)



RUNNING CASE: cnn_lstm
predict_test=False

=== cnn_lstm | Fold 0: train [12, 13, 14] -> validate 15 (train=315,927, valid=101,632) ===
-- seed 42 --
   ep  1/30 start (train_batches=618, valid_batches=199)


cnn_lstm seed42 ep1/30 train:   0%|          | 0/618 [00:00<?, ?it/s]

cnn_lstm seed42 ep1/30 valid:   0%|          | 0/199 [00:00<?, ?it/s]

   ep  1/30  loss=0.1902  val_auc=0.8647  (52.1s)
   ep  2/30 start (train_batches=618, valid_batches=199)


cnn_lstm seed42 ep2/30 train:   0%|          | 0/618 [00:00<?, ?it/s]

...

cnn_lstm seed44 ep20/30 valid:   0%|          | 0/175 [00:00<?, ?it/s]

   ep 20/30  loss=0.0582  val_auc=0.8831  (87.6s)
   ep 21/30 start (train_batches=979, valid_batches=175)


cnn_lstm seed44 ep21/30 train:   0%|          | 0/979 [00:00<?, ?it/s]

cnn_lstm seed44 ep21/30 valid:   0%|          | 0/175 [00:00<?, ?it/s]

   ep 21/30  loss=0.0566  val_auc=0.8821  (87.7s)
   early stop at epoch 21
   fold AUC (seed-avg) = 0.9111

=== cnn_lstm OOF AUC (validated months only) = 0.8995 ===
   per-fold: [(15, 0.8939786610345147), (16, 0.8947514483804436), (17, 0.9111304026986573)]
Saved cnn_hybrid_outputs/oof_cnn_lstm.csv
Skipped test prediction. Set PREDICT_TEST=True for final submission inference.
Saved cnn_hybrid_outputs/summary_cnn_lstm.csv


### Case 2: CNN-BiGRU-Feature-Attention

Feature attention weights critical transaction features, CNN extracts local feature interactions, and a bidirectional GRU models the temporal sequence. Test prediction is skipped in this tuning run.


In [ ]:
cnn_bigru_attention_summary = run_experiment_case("cnn_bigru_attention", predict_test=False)


### Case 3: CNN-LSTM-Feature-Attention

Feature attention weights critical transaction features, CNN extracts local feature interactions, and a bidirectional LSTM models the temporal sequence. Test prediction is skipped in this tuning run.


In [11]:
cnn_lstm_attention_summary = run_experiment_case("cnn_lstm_attention", predict_test=False)



RUNNING CASE: cnn_lstm_attention
predict_test=False

=== cnn_lstm_attention | Fold 0: train [12, 13, 14] -> validate 15 (train=315,927, valid=101,632) ===
-- seed 42 --
   ep  1/30 start (train_batches=618, valid_batches=199)


cnn_lstm_attention seed42 ep1/30 train:   0%|          | 0/618 [00:00<?, ?it/s]

cnn_lstm_attention seed42 ep1/30 valid:   0%|          | 0/199 [00:00<?, ?it/s]

   ep  1/30  loss=0.1955  val_auc=0.8738  (54.0s)
   ep  2/30 start (train_batches=618, valid_batches=199)


cnn_lstm_attention seed42 ep2/30 train:   0%|          | 0/618 [00:00<?, ?it/s]

...

cnn_lstm_attention seed44 ep17/30 valid:   0%|          | 0/175 [00:00<?, ?it/s]

   ep 17/30  loss=0.0446  val_auc=0.8948  (90.8s)
   ep 18/30 start (train_batches=979, valid_batches=175)


cnn_lstm_attention seed44 ep18/30 train:   0%|          | 0/979 [00:00<?, ?it/s]

cnn_lstm_attention seed44 ep18/30 valid:   0%|          | 0/175 [00:00<?, ?it/s]

   ep 18/30  loss=0.0432  val_auc=0.9025  (90.8s)
   early stop at epoch 18
   fold AUC (seed-avg) = 0.9244

=== cnn_lstm_attention OOF AUC (validated months only) = 0.9149 ===
   per-fold: [(15, 0.9071794710121238), (16, 0.9219036916974034), (17, 0.9244490851382445)]
Saved cnn_hybrid_outputs/oof_cnn_lstm_attention.csv
Skipped test prediction. Set PREDICT_TEST=True for final submission inference.
Saved cnn_hybrid_outputs/summary_cnn_lstm_attention.csv


### Optional Combined Summary

Run this after one or more case cells to display the summaries created in the current kernel session.


In [12]:
summary = pd.DataFrame(experiment_summaries).sort_values('overall_oof_auc', ascending=False)
summary_path = OUTPUT_DIR / 'summary_cnn_hybrid_experiments.csv'
summary.to_csv(summary_path, index=False)
print(f"Saved {summary_path}")
try:
    display(summary)
except NameError:
    print(summary)


Saved cnn_hybrid_outputs/summary_cnn_hybrid_experiments.csv


,case,overall_oof_auc,fold_aucs,oof_path,test_path,predict_test,n_seeds,n_folds,n_resblocks,skip_padded_cnn_rows,use_packed_rnn,ae_filter_enabled
2,cnn_lstm_attention,0.914906,"[(15, 0.9071794710121238), (16, 0.921903691697...",cnn_hybrid_outputs/oof_cnn_lstm_attention.csv,,False,3,3,2,True,True,False
1,cnn_bigru_attention,0.913065,"[(15, 0.9041888283376448), (16, 0.915409442074...",cnn_hybrid_outputs/oof_cnn_bigru_attention.csv,,False,3,3,2,True,True,False
0,cnn_lstm,0.899494,"[(15, 0.8939786610345147), (16, 0.894751448380...",cnn_hybrid_outputs/oof_cnn_lstm.csv,,False,3,3,2,True,True,False
